# CSCI4052 GROUP FINAL PROJECT 

MEMBERS: Tony Akinniranye, Lukas Fenkam, jaathavan

For our project, we dedided to make a sign language interpreter


In [8]:
import fiftyone as fo
import fiftyone.utils.huggingface as fouh


In [ ]:
'''
# 1. Load the dataset (either from Hugging Face or from disk)

'''

RE_DOWNLOAD_DATASET = False  # set to True if you want to re-download the dataset, False to load from disk

if RE_DOWNLOAD_DATASET:
    fo.delete_dataset("wlasl_top20")  

    dataset = fouh.load_from_hub(
        "Voxel51/WLASL",
        max_samples=5000,       # sample size, hugging face api max is 5000, actual dataset has  11,980 samples
        persistent=True,        # saves to disk so you don't re-download
        name="wlasl_top2"            # names it so you can reload it later
    )

else:
    dataset = fo.load_dataset("wlasl_top2")

Loading dataset
Importing samples...
 100% |███████████████| 5000/5000 [190.4ms elapsed, 0s remaining, 26.3K samples/s]     
Importing frames...
 100% |███████████| 351245/351245 [9.1s elapsed, 0s remaining, 39.5K samples/s]      
Migrating dataset 'wlasl' to v1.13.4


100%|██████████| 1/1 [00:03<00:00,  3.33s/it]


In [ ]:
'''
# 2. Exploratory Data Analysis 

    Start by understanding the structure of the dataset, the distribution of classes, and any potential issues (e.g. class imbalance, missing data) that may affect model training. 
    This will inform the preprocessing and modeling decisions later on.
'''

# Exploratory data analysis
from collections import Counter

# basic dataset info
print(f"Total Samples: {len(dataset)}")
print(f"Fields: {list(dataset.get_field_schema().keys())}\n")


# get all the unique labels 
distinct_labels = dataset.distinct("gloss.label")
print(f"Unique Labels: {len(distinct_labels)}")
print(f"Sample Labels: {distinct_labels[:10]}\n")


# Check the structure of a single sample first
# get all labels (one per sample, with duplicates)
raw_labels = dataset.values("gloss.label")
print(f"Total labels retrieved: {len(raw_labels)}")
print(f"Sample of raw labels: {raw_labels[:10]}")

# Class distribution
class_counts = Counter(raw_labels)
print(f"\nTop 10 most common signs:")
for label, count in class_counts.most_common(10):
    print(f"  {label}: {count} samples")

print(f"\nBottom 10 least common signs:")
for label, count in class_counts.most_common()[-20:]:
    print(f"  {label}: {count} samples")

print(f"\nAvg samples per class: {len(raw_labels) / len(set(raw_labels)):.1f}")


Total Samples: 5000
Fields: ['id', 'filepath', 'tags', 'metadata', 'created_at', 'last_modified_at', 'bounding_box', 'gloss']

Unique Labels: 931
Sample Labels: ['a', 'a lot', 'abdomen', 'able', 'about', 'above', 'accent', 'accept', 'accident', 'accomplish']

Total labels retrieved: 5000
Sample of raw labels: ['abdomen', 'abdomen', 'abdomen', 'abdomen', 'abdomen', 'able', 'able', 'able', 'able', 'able']

Top 10 most common signs:
  before: 15 samples
  computer: 14 samples
  cool: 13 samples
  cousin: 13 samples
  go: 13 samples
  accident: 12 samples
  change: 12 samples
  drink: 12 samples
  bowling: 11 samples
  cold: 11 samples

Bottom 10 least common signs:
  grandma: 3 samples
  grow up: 3 samples
  haircut: 3 samples
  hang up: 3 samples
  h: 3 samples
  hill: 3 samples
  hippopotamus: 3 samples
  honey: 3 samples
  i: 3 samples
  individual: 3 samples
  battery: 2 samples
  boots: 2 samples
  careless: 2 samples
  curtain: 2 samples
  devil: 2 samples
  either: 2 samples
  excu

In [13]:

# get top 20 classes
top_20 = [label for label, count in class_counts.most_common(20)]
print("Top 20 classes:", top_20)

# filter dataset to only those classes
view = dataset.filter_labels("gloss", fo.ViewField("label").is_in(top_20))
print(f"Filtered samples: {len(view)}")

# verify new distribution
filtered_labels = view.values("gloss.label")
filtered_counts = Counter(filtered_labels)
for label, count in filtered_counts.most_common():
    print(f"  {label}: {count} samples")
    
print(f"Avg samples per class after filtering: {len(filtered_labels) / len(set(filtered_labels)):.1f}")

Top 20 classes: ['before', 'computer', 'cool', 'cousin', 'go', 'accident', 'change', 'drink', 'bowling', 'cold', 'delay', 'bar', 'bed', 'call', 'candy', 'champion', 'check', 'deaf', 'far', 'help']
Filtered samples: 227
  before: 15 samples
  computer: 14 samples
  cool: 13 samples
  cousin: 13 samples
  go: 13 samples
  accident: 12 samples
  change: 12 samples
  drink: 12 samples
  bowling: 11 samples
  cold: 11 samples
  delay: 11 samples
  bar: 10 samples
  bed: 10 samples
  call: 10 samples
  candy: 10 samples
  champion: 10 samples
  check: 10 samples
  deaf: 10 samples
  far: 10 samples
  help: 10 samples
Avg samples per class after filtering: 11.3


In [ ]:
# Export top 20 classes to a folder for training
import os

export_dir = "C:/Users/tonya/wlasl_top2"

view.export(
    export_dir=export_dir,
    dataset_type=fo.types.VideoClassificationDirectoryTree,
    label_field="gloss",
)

print(f"Exported dataset to {export_dir}")
print(f"Total samples exported: {len(view)}")

 100% |█████████████████| 227/227 [3.1s elapsed, 0s remaining, 62.7 samples/s]      
Exported dataset to C:/Users/tonya/wlasl
Total samples exported: 227
